## Part 2 Datasets/DataFrames: Spark ML and Pipelines
Convert the review texts to a classic vector space representation with TFIDF-weighted features based on the Spark DataFrame/Dataset API by building a transformation pipeline. The primary goal of this part is the preparation of the pipeline for Part 3 (see below). Note: although parts of this pipeline will be very similar to Assignment 1 or Part 1 above, do not expect to obtain identical results or have access to all intermediate outputs to compare the individual steps.

Use built-in functions for tokenization to unigrams at whitespaces, tabs, digits, and the delimiter characters ()[]{}.!?,;:+=-_"'`~#@&*%€$§\/, casefolding, stopword removal, TF-IDF calculation, and chi square selection ) (using 2000 top terms overall). Write the terms selected this way to a file output_ds.txt and compare them with the terms selected in Assignment 1. Describe your observations briefly in the submission report (see Part 3).

[Provided link for ML pipeline](https://spark.apache.org/docs/latest/ml-pipeline.html)  
[Provided link for feature extraction](https://spark.apache.org/docs/latest/ml-features.html)

## Imports

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.feature import StopWordsRemover
from pyspark.ml.feature import IDF

from pyspark.ml.feature import RegexTokenizer

from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import CountVectorizer
from pyspark.ml.feature import UnivariateFeatureSelector

from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.classification import OneVsRest
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.ml.feature import Normalizer

from datetime import datetime

In [2]:
import warnings
warnings.filterwarnings(action='once')

## Initialize Spark

In [3]:
# Initialize Spark context and session
# Stop existing SparkContext if it's running
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()

conf = SparkConf().setAppName("Part2")
sc = SparkContext(conf=conf)
spark = SparkSession(sc)

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/usr/lib/spark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/usr/lib/hadoop/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.
25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4045. Attempting port 4046.
25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4046. Attempting port 4047.
25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4047. Attempting port 4048.
25/05/12 09:07:19 WARN Utils: Service 'SparkUI' could not bind on port 4048. Attempting port 4049.
25/05/12 09:07:21 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploadi

In [4]:
spark

### Importing data
Stopwords file as well as test data

In [5]:
# Define the stopwords file 
stop_file = "stopwords.txt"

# Load stopwords into a set
with open(stop_file, "r") as f:
    stopwords = set(f.read().strip().split())
    
# Load and preprocess the Amazon reviews dataset devset not the full one!
input_file = "hdfs:///user/dic25_shared/amazon-reviews/full/reviews_devset.json"
reviews_df = spark.read.json(input_file)

In [6]:
reviews_df = reviews_df.select("category", "reviewText")

# Show the DataFrame with selected columns
reviews_df.show()

+--------------------+--------------------+
|            category|          reviewText|
+--------------------+--------------------+
|Patio_Lawn_and_Garde|This was a gift f...|
|Patio_Lawn_and_Garde|This is a very ni...|
|Patio_Lawn_and_Garde|The metal base wi...|
|Patio_Lawn_and_Garde|For the most part...|
|Patio_Lawn_and_Garde|This hose is supp...|
|Patio_Lawn_and_Garde|This tool works v...|
|Patio_Lawn_and_Garde|This product is a...|
|Patio_Lawn_and_Garde|I was excited to ...|
|Patio_Lawn_and_Garde|I purchased the L...|
|Patio_Lawn_and_Garde|Never used a manu...|
|Patio_Lawn_and_Garde|Good price. Good ...|
|Patio_Lawn_and_Garde|I have owned the ...|
|Patio_Lawn_and_Garde|I had "won" a sim...|
|Patio_Lawn_and_Garde|The birds ate all...|
|Patio_Lawn_and_Garde|Bought last summe...|
|Patio_Lawn_and_Garde|I knew I had a mo...|
|Patio_Lawn_and_Garde|I was a little wo...|
|Patio_Lawn_and_Garde|I have used this ...|
|Patio_Lawn_and_Garde|I actually do not...|
|Patio_Lawn_and_Garde|Just what 

## Create RegexTokenizer

In [7]:
# define regex-based tokenizer to split the text
pattern = r'\s+|\d+|[(){}\[\].!?,;:+=_"\'`~#@&*%€$§\\/\-]'
regexTokenizer = RegexTokenizer(inputCol="reviewText", outputCol="words", pattern=pattern)

## Create StopWordsRemover

In [8]:
# remove standard stop words from tokenized words
remover = StopWordsRemover(stopWords = list(stopwords), inputCol="words", outputCol="filtered_words")

## Create CountVectorizer

In [9]:
# convert filtered words into a sparse vector using CountVectorizer
countV = CountVectorizer(inputCol="filtered_words", outputCol="rawFeatures").setMinTF(1).setMinDF(3).setVocabSize(7500)

## Create IDF model

In [10]:
# Apply inverse document frequency to weight the words
idf = IDF(inputCol="rawFeatures", outputCol="features")

## Create StringIndexer

In [11]:
# Apply StringIndexer to convert categorical column to numerical
indexer = StringIndexer(inputCol="category", outputCol="label")

## Create Selector
Here, we add a SelectionThreshold of 2000 to have the top 2000 terms overall (as given by the assignment).

In [12]:
selector = UnivariateFeatureSelector(
    featuresCol="features",outputCol="selectedFeatures",
    labelCol="label",  selectionMode="numTopFeatures",
)
selector.setFeatureType("categorical").setLabelType("categorical").setSelectionThreshold(2000)

UnivariateFeatureSelector_fac86b4852a3

## Create Pipeline

In [13]:
pipeline = Pipeline(stages=[regexTokenizer, remover, countV, idf, indexer, selector])

In [14]:
model = pipeline.fit(reviews_df)

In [15]:
result = model.transform(reviews_df)

In [16]:
result.select("selectedFeatures").show(truncate=True)

+--------------------+
|    selectedFeatures|
+--------------------+
|(2000,[2,3,7,8,35...|
|(2000,[0,1,3,21,3...|
|(2000,[4,10,174,3...|
|(2000,[1,3,4,9,10...|
|(2000,[12,29,101,...|
|(2000,[0,3,4,8,11...|
|(2000,[18,112,175...|
|(2000,[6,21,32,36...|
|(2000,[3,4,5,6,40...|
|(2000,[6,8,38,78,...|
|(2000,[1,13,226],...|
|(2000,[5,17,33,40...|
|(2000,[1,11,28,35...|
|(2000,[40,144,339...|
|(2000,[0,3,7,9,11...|
|(2000,[8,26,57,80...|
|(2000,[1,15,120,1...|
|(2000,[2,3,221,26...|
|(2000,[4,10,16,20...|
|(2000,[0,18,30,42...|
+--------------------+
only showing top 20 rows



## Get top 2000 terms

In [17]:
# Extract vocabulary from CountVectorizer
vocabulary = model.stages[2].vocabulary

# Map selected feature indices to terms
selected_terms = [vocabulary[i] for i in model.stages[-1].selectedFeatures]

In [18]:
len(selected_terms)

2000

In [19]:
vocabulary[:30] == selected_terms[:30]

False

In [20]:
type(selected_terms)

list

In [21]:
with open("output_ds.txt", "w") as f:
    for term in selected_terms:
        f.write(term + "\n")

# Part 3

In this part, we need to train a text classifier from the features extracted in Part 2. The goal is to create a model that can predict the product category from a review's text.

To this end, we need to extend the pipeline from Part 2 such that a Support Vector Machine classifier is trained. Since we are dealing with multi-class problems, we use a one-vs-rest classifier and apply vector length normalization before feeding the feature vectors into the classifier (use Normalizer with L2 norm).

To follow best practices for machine learning experiment design we investigate the effects of parameter settings and ...

- Split the review data into training, validation and test set.

- Make experiments reproducible.

- Use a grid search for parameter optimization:

- Compare chi square overall top 2000 filtered features

- Compare different SVM settings by varying the regularization parameter (choose 3 different values), standardization of training features (2 values), and maximum number of iterations (2 values).

- Use the MulticlassClassificationEvaluator to estimate performance of our trained classifiers on the test set with the F1 measure as criterion.

## Split into Train/Test/Val and set seed

In [22]:
(reviews_train, reviews_test) = reviews_df.randomSplit([0.9, 0.1], 123)

In [23]:
reviews_train.count()

71031

In [24]:
reviews_test.count()

7798

## Extend Pipeline

In [25]:
normalizer = Normalizer(inputCol="selectedFeatures", outputCol="normalized_features")

In [26]:
svm = LinearSVC(featuresCol="normalized_features")
ovr = OneVsRest(classifier=svm)

In [27]:
extended_pipeline = Pipeline(stages=[regexTokenizer, remover, countV, idf, indexer, selector, normalizer, ovr])

## Define Evaluator

In [28]:
evaluator = MulticlassClassificationEvaluator(metricName="f1")

## Perform Grid Search using different SVM settings

Create a param grid to train and evaluate pipeline. Use TrainValidationSplit for single model per combination.

In [29]:
param_grid = ParamGridBuilder() \
    .addGrid(svm.regParam, [0.1, 0.01, 0.001]) \
    .addGrid(svm.maxIter, [10, 20]) \
    .addGrid(svm.standardization, [True, False]) \
    .addGrid(selector.selectionThreshold, [2000, 250]) \
    .build()
#
validator = TrainValidationSplit(estimator=extended_pipeline, estimatorParamMaps=param_grid, evaluator=evaluator, trainRatio = 0.7, parallelism = 4, seed=123)

We perform a grid search on a subset of the training data to approximate the best parameters. Following we fit it to train the model and then and we use the best model to compute the predictions. 

In [30]:
# start and stop time while model performs the grid search
start_time = datetime.now()
cv_model = validator.fit(reviews_train)
end_time = datetime.now()

/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=70, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 58272), raddr=('127.0.0.1', 38201)>
  self._sock = None
/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=70, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 46322), raddr=('127.0.0.1', 39133)>
  self._sock = None
/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=77, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 46198), raddr=('127.0.0.1', 40257)>
  self._sock = None
[Stage 82:==> (1 + 1) / 2][Stage 85:==> (1 + 0) / 2][Stage 86:==> (1 + 0) / 2]  

25/05/12 09:09:52 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/05/12 09:09:52 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS


25/05/12 09:09:56 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
25/05/12 09:09:56 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS


/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=80, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 57878), raddr=('127.0.0.1', 36399)>
  self._sock = None
/usr/lib64/python3.9/multiprocessing/pool.py:265: ResourceWarning: unclosed running multiprocessing pool <multiprocessing.pool.ThreadPool state=RUN pool_size=1>
  _warn(f"unclosed running multiprocessing pool {self!r}",


25/05/12 09:21:26 WARN DAGScheduler: Broadcasting large task binary with size 1826.7 KiB
25/05/12 09:21:26 WARN DAGScheduler: Broadcasting large task binary with size 1826.7 KiB
25/05/12 09:21:27 WARN DAGScheduler: Broadcasting large task binary with size 1826.7 KiB
25/05/12 09:21:27 WARN DAGScheduler: Broadcasting large task binary with size 1826.7 KiB


...

/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=72, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 53270), raddr=('127.0.0.1', 35635)>
  self._sock = None
/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=72, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 43018), raddr=('127.0.0.1', 33049)>
  self._sock = None
/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=75, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 56016), raddr=('127.0.0.1', 39189)>
  self._sock = None
/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=82, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 39188), raddr=('127.0.0.1', 41233)>
  self._sock = None
                                                                                

25/05/12 10:31:04 WARN DAGScheduler: Broadcasting large task binary with size 1826.7 KiB
25/05/12 10:31:04 WARN DAGScheduler: Broadcasting large task binary with size 1826.7 KiB


[Stage 19057:>              (0 + 2) / 2][Stage 19059:>              (0 + 0) / 2]

25/05/12 10:31:05 WARN DAGScheduler: Broadcasting large task binary with size 1826.7 KiB
25/05/12 10:31:05 WARN DAGScheduler: Broadcasting large task binary with size 1826.7 KiB


/usr/lib64/python3.9/socket.py:787: ResourceWarning: unclosed <socket.socket fd=72, family=AddressFamily.AF_INET, type=SocketKind.SOCK_STREAM, proto=6, laddr=('127.0.0.1', 37460), raddr=('127.0.0.1', 38811)>
  self._sock = None
/usr/lib64/python3.9/multiprocessing/pool.py:265: ResourceWarning: unclosed running multiprocessing pool <multiprocessing.pool.ThreadPool state=RUN pool_size=4>
  _warn(f"unclosed running multiprocessing pool {self!r}",


In [31]:
print("start time:", start_time)
print("end time:", end_time)
print("time elapsed:", end_time - start_time)

print("\nno. reviews used to train:", reviews_train.count())
print("no. reviews used to test:", reviews_test.count())

start time: 2025-05-12 09:08:36.597873
end time: 2025-05-12 10:36:42.775489
time elapsed: 1:28:06.177616



no. reviews used to train: 71031
no. reviews used to test: 7798


In [32]:
# Get the best model
bestModel = cv_model.bestModel

# Show the best parameters
print("Best Params:")
print("RegParam:", bestModel.stages[-1].models[0].getOrDefault("regParam"))
print("MaxIter:", bestModel.stages[-1].models[0].getOrDefault("maxIter"))
print("Standardization:", bestModel.stages[-1].models[0].getOrDefault("standardization"))
print("P value:", bestModel.stages[-3].getOrDefault("selectionThreshold"))

Best Params:
RegParam: 0.01
MaxIter: 10
Standardization: False
P value: 2000.0


In [33]:
# get the predictions
predictions = cv_model.transform(reviews_test)
# evaluate on the f1 score and print the result
f1_score = evaluator.evaluate(predictions)
print(f"F1 Score: {f1_score}")

25/05/12 10:36:52 WARN DAGScheduler: Broadcasting large task binary with size 1823.5 KiB


[Stage 19636:===========================>                           (1 + 1) / 2]

F1 Score: 0.630127489411732


In [34]:
spark.stop()